In [17]:
import requests

def get_kiel_bounding_box(factor=1):
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": "Kiel, Germany",
        "format": "json",
        "limit": 1
    }
    headers = {"User-Agent": "Data Science Project CAU WS25/26 Group 16"}

    response = requests.get(url, params=params, headers=headers)
    data = response.json()

    bb = data[0]["boundingbox"]
    lat_min = float(bb[0])
    lat_max = float(bb[1])
    lon_min = float(bb[2])
    lon_max = float(bb[3])

    # Increase boundingbox size by "factor"
    lat_center = (lat_min + lat_max) / 2
    lon_center = (lon_min + lon_max) / 2
    lat_span   = (lat_max - lat_min) / 2
    lon_span   = (lon_max - lon_min) / 2

    return {
        "lat_min": lat_center - lat_span * factor,
        "lat_max": lat_center + lat_span * factor,
        "lon_min": lon_center - lon_span * factor,
        "lon_max": lon_center + lon_span * factor,
    }


def is_near_kiel(lat, lon, bb):
    return bb["lat_min"] <= lat <= bb["lat_max"] and bb["lon_min"] <= lon <= bb["lon_max"]

In [18]:
import os
import polars as pl
from pyproj import Transformer
from collections import defaultdict
#from kiel_utils import get_kiel_bounding_box, is_near_kiel

# ─────────────────────────────────────────────
# 1. UTM32 (ETRS89) → WGS84 (lat/lon)
# ─────────────────────────────────────────────
transformer = Transformer.from_crs("EPSG:25832", "EPSG:4326", always_xy=False)

def utm32_to_latlon(easting, northing):
    lat, lon = transformer.transform(easting, northing)
    return lat, lon


# ─────────────────────────────────────────────
# 2. Hilfsfunktion: deutsche Dezimalzahl parsen
#    z.B. "594121,056" → 594121.056
# ─────────────────────────────────────────────
def parse_german_float(value):
    return float(str(value).replace(".", "").replace(",", "."))


# ─────────────────────────────────────────────
# 3. Hauptprogramm
# ─────────────────────────────────────────────
def main():
    # Pfad zum Oberordner anpassen!
    BASE_DIR = "../../data/BASt/AggregatedDataForSH"
    YEARS    = [2021, 2022, 2023, 2024, 2025]
    MONTHS   = [f"{m:02d}" for m in range(1, 13)]
    VARIANTS = ["A", "B"]

    # Kiel Bounding Box laden
    print("Lade Kiel Bounding Box von Nominatim...")
    bb = get_kiel_bounding_box(factor=3)
    print(f"  Lat: {bb['lat_min']:.4f} – {bb['lat_max']:.4f}")
    print(f"  Lon: {bb['lon_min']:.4f} – {bb['lon_max']:.4f}\n")

    # Dictionary: Dauerzaehlstellennummer → Set von Monaten, in denen sie vorhanden war
    # Format: { 1111: {"2021-01", "2021-02", ...}, ... }
    station_months = defaultdict(set)

    # Gesammelte gefilterte Roh-DataFrames
    alle_roh_kiel = []

    for year in YEARS:
        for variant in VARIANTS:
            folder = os.path.join(BASE_DIR, f"Data {year}", f"SH_{year}_{variant}_S")

            for month in MONTHS:
                meta_file = os.path.join(folder, f"Meta_{year}_{month}_{variant}_S.csv")
                roh_file  = os.path.join(folder, f"Roh_{year}_{month}_{variant}_S.csv")

                # Dateien überspringen falls nicht vorhanden
                if not os.path.exists(meta_file):
                    print(f"  [FEHLT] {meta_file}")
                    continue
                if not os.path.exists(roh_file):
                    print(f"  [FEHLT] {roh_file}")
                    continue

                print(f"Verarbeite: {year}-{month} Variante {variant}")

                # Meta einlesen
                meta_df = pl.read_csv(meta_file, infer_schema_length=0)  # alles als String

                # Koordinaten umwandeln und Kiel-Filter anwenden
                kiel_nummern = []

                for row in meta_df.iter_rows(named=True):
                    try:
                        easting  = parse_german_float(row["Koordinaten_UTM32_E"])
                        northing = parse_german_float(row["Koordinaten_UTM32_N"])
                        lat, lon = utm32_to_latlon(easting, northing)

                        if is_near_kiel(lat, lon, bb):
                            nr = int(row["Dauerzaehlstellennummer"])
                            kiel_nummern.append(nr)

                            # Monat im Dictionary vermerken
                            monat_key = f"{year}-{month}"
                            station_months[nr].add(monat_key)

                    except Exception as e:
                        print(f"    Fehler bei Zeile {row.get('Dauerzaehlstellennummer', '?')}: {e}")

                print(f"  → {len(kiel_nummern)} Kieler Stationen gefunden: {kiel_nummern}")

                # Roh-Daten einlesen und auf Kieler Stationen filtern
                if kiel_nummern:
                    roh_df = pl.read_csv(roh_file, infer_schema_length=0)  # alles als String

                    # "Zst" Spalte = Dauerzaehlstellennummer (als int vergleichen)
                    roh_kiel = (
                        roh_df
                        .with_columns(pl.col("Zst").str.strip_chars().cast(pl.Int64, strict=False).alias("Zst_int"))
                        .filter(pl.col("Zst_int").is_in(kiel_nummern))
                        .drop("Zst_int")
                        .with_columns([
                            pl.lit(year).alias("Jahr"),
                            pl.lit(month).alias("Monat"),
                            pl.lit(variant).alias("Variante"),
                        ])
                    )

                    alle_roh_kiel.append(roh_kiel)
                    print(f"  → {roh_kiel.height} Roh-Zeilen behalten")

    # ─────────────────────────────────────────────
    # 4. Ergebnisse speichern
    # ─────────────────────────────────────────────

    # 4a. Alle Roh-Daten zusammenführen und speichern
    gesamt_df = None
    if alle_roh_kiel:
        gesamt_df = pl.concat(alle_roh_kiel, how="diagonal")  # diagonal: fehlende Spalten mit null auffüllen
        gesamt_df.write_csv("../../data/BASt/AggregatedDataForKiel/kiel_roh_gefiltert.csv")
        print(f"\nGesamt-Roh-Daten gespeichert: kiel_roh_gefiltert.csv ({gesamt_df.height} Zeilen)")

    # 4b. Station-Monats-Dictionary ausgeben und speichern
    print("\n── Station-Präsenz-Dictionary ──")
    station_presence_rows = []
    for nr, monate in sorted(station_months.items()):
        unique_monate = sorted(monate)
        print(f"  Station {nr}: {len(unique_monate)} Monate → {unique_monate}")
        station_presence_rows.append({
            "Dauerzaehlstellennummer": nr,
            "Anzahl_Monate": len(unique_monate),
            "Monate": ", ".join(unique_monate)
        })

    presence_df = pl.DataFrame(station_presence_rows)
    presence_df.write_csv("../../data/BASt/AggregatedDataForKiel/kiel_station_praesenz.csv")
    print("\nStation-Präsenz gespeichert: kiel_station_praesenz.csv")

    # 4c. Stationen mit fehlenden Monaten aufzeigen
    max_monate = len(YEARS) * 12  # 60 Monate
    print("\n── Stationen mit fehlenden Monaten ──")
    for nr, monate in sorted(station_months.items()):
        if len(monate) < max_monate:
            print(f"  Station {nr}: nur {len(monate)}/{max_monate} Monate vorhanden")

    return station_months, gesamt_df


if __name__ == "__main__":
    station_months, roh_df = main()

Lade Kiel Bounding Box von Nominatim...
  Lat: 54.0681 – 54.6159
  Lon: 9.8472 – 10.4043

Verarbeite: 2021-01 Variante A
  → 5 Kieler Stationen gefunden: [1104, 1106, 1156, 1162, 1194]
  → 3720 Roh-Zeilen behalten
Verarbeite: 2021-02 Variante A
  → 5 Kieler Stationen gefunden: [1104, 1106, 1156, 1162, 1194]
  → 3360 Roh-Zeilen behalten
Verarbeite: 2021-03 Variante A
  → 5 Kieler Stationen gefunden: [1104, 1106, 1156, 1162, 1194]
  → 3720 Roh-Zeilen behalten
Verarbeite: 2021-04 Variante A
  → 5 Kieler Stationen gefunden: [1104, 1106, 1156, 1162, 1194]
  → 3600 Roh-Zeilen behalten
Verarbeite: 2021-05 Variante A
  → 5 Kieler Stationen gefunden: [1104, 1106, 1156, 1162, 1194]
  → 3720 Roh-Zeilen behalten
Verarbeite: 2021-06 Variante A
  → 5 Kieler Stationen gefunden: [1104, 1106, 1156, 1162, 1194]
  → 3600 Roh-Zeilen behalten
Verarbeite: 2021-07 Variante A
  → 5 Kieler Stationen gefunden: [1104, 1106, 1156, 1162, 1194]
  → 3720 Roh-Zeilen behalten
Verarbeite: 2021-08 Variante A
  → 5 Kiele